# Week 2 – Mini Data Quality Audit

*Dataset: `week2_customer_transactions_messy.csv` | 2026-04-26*

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print("pandas:", pd.__version__, "| numpy:", np.__version__)

In [ ]:
data_path = Path('week2_customer_transactions_messy.csv')
df = pd.read_csv(data_path)
print(f"Dataset loaded: {data_path}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
df

## Task 1 – Dataset Description

### Description

The dataset has 11 rows and 9 columns of customer payment transactions across Germany, France, the Netherlands, and the US. Each row is one transaction with an ID, customer reference, date, amount, currency, payment method, status, and when the record was last updated.

It looks like it comes from a payments or e-commerce backend. Likely uses: revenue reporting, payment channel tracking, and flagging unusual activity.

Columns at a glance:

| Column | Description |
|---|---|
| `transaction_id` | unique key per transaction |
| `customer_id` | links to a customer account |
| `transaction_date` | when the transaction happened |
| `amount` | value in the stated currency |
| `currency` | should be an ISO 4217 code (EUR, USD…) |
| `payment_method` | card / bank_transfer / cash |
| `status` | completed / pending / cancelled |
| `region` | ISO 3166-1 alpha-2 country code |
| `last_updated` | when the record was last modified |

In [ ]:
print("data types:")
print(df.dtypes)
print("\nmissing values:")
print(df.isnull().sum())
print("\nunique values per column:")
for col in df.columns:
    print(f"  {col}: {df[col].dropna().unique().tolist()}")

## Task 2 – Data Quality Issues by Dimension

In [ ]:
def is_missing(series):
    return series.isna() | (series.astype(str).str.strip() == '')

# uniqueness
dup_mask = df.duplicated(subset=['transaction_id'], keep=False)

# validity
amount_num  = pd.to_numeric(df['amount'], errors='coerce')
date_parsed = pd.to_datetime(df['transaction_date'].str.strip(), errors='coerce', format='mixed')
VALID_CURRENCIES = {'EUR', 'USD', 'GBP', 'CHF', 'JPY', 'CAD', 'AUD', 'NZD'}
invalid_currency = ~df['currency'].str.upper().str.strip().isin(VALID_CURRENCIES)
extreme_amount   = amount_num > 100_000

# consistency
non_iso_date = (
    df['transaction_date'].str.contains('/', na=False) |
    df['transaction_date'].str.match(r'^\d{2}-\d{2}-\d{4}', na=False)
)
region_not_upper  = df['region'].notna() & (df['region'] != df['region'].str.upper())
payment_not_lower = df['payment_method'].notna() & (df['payment_method'] != df['payment_method'].str.lower())

# integrity – flag records where last_updated is >60 days after transaction_date
lu_parsed  = pd.to_datetime(df['last_updated'], errors='coerce', format='mixed')
update_gap = (lu_parsed - date_parsed).dt.days
suspicious_lag = (update_gap > 60) & update_gap.notna()

issue_rows = [
    ['Missing customer_id',              'Completeness', int(is_missing(df['customer_id']).sum()),    'High'],
    ['Missing amount',                   'Completeness', int(is_missing(df['amount']).sum()),          'High'],
    ['Missing payment_method',           'Completeness', int(is_missing(df['payment_method']).sum()),  'Medium'],
    ['Missing last_updated',             'Completeness', int(is_missing(df['last_updated']).sum()),    'Low'],
    ['Duplicate transaction_id',         'Uniqueness',   int(dup_mask.sum()),                          'High'],
    ['Negative amount',                  'Validity',     int((amount_num < 0).sum()),                  'High'],
    ['Invalid/impossible date',          'Validity',     int(date_parsed.isna().sum()),                'High'],
    ['Non-ISO currency (e.g. EURO)',     'Validity',     int(invalid_currency.sum()),                  'Medium'],
    ['Extreme outlier (amount > 100k)',  'Validity',     int(extreme_amount.sum()),                    'Medium'],
    ['Inconsistent date format',         'Consistency',  int(non_iso_date.sum()),                      'Medium'],
    ['Region not uppercase',             'Consistency',  int(region_not_upper.sum()),                  'Low'],
    ['Payment method not lowercase',     'Consistency',  int(payment_not_lower.sum()),                 'Low'],
    ['Suspicious update lag (>60 days)', 'Integrity',    int(suspicious_lag.sum()),                    'Medium'],
]
issue_table = pd.DataFrame(issue_rows, columns=['Issue', 'Dimension', 'Affected Rows', 'Severity'])
issue_table

## Task 3 – KPI Calculations

In [ ]:
n_rows = len(df)
n_cols = len(df.columns)

# KPI 1 – Completeness Rate
total_missing     = df.isnull().sum().sum()
completeness_rate = 1 - (total_missing / (n_rows * n_cols))

# KPI 2 – Duplication Rate
dup_rate = df.duplicated(subset=['transaction_id']).sum() / n_rows

# KPI 3 – Validity Error Rate
has_validity_issue  = (amount_num < 0) | date_parsed.isna() | invalid_currency | extreme_amount
validity_error_rate = has_validity_issue.mean()

# KPI 4 – Format Consistency Score
format_issue       = non_iso_date | region_not_upper | payment_not_lower
format_consistency = 1 - format_issue.mean()

kpi_df = pd.DataFrame([
    ['Completeness Rate',        completeness_rate,   '>= 98%', completeness_rate >= 0.98],
    ['Duplication Rate',         dup_rate,            '0%',     dup_rate == 0.0],
    ['Validity Error Rate',      validity_error_rate, '0%',     validity_error_rate == 0.0],
    ['Format Consistency Score', format_consistency,  '>= 95%', format_consistency >= 0.95],
], columns=['KPI', 'Value', 'Target', 'Meets Target'])
kpi_df['Value (%)']    = kpi_df['Value'].apply(lambda x: f'{x*100:.1f}%')
kpi_df['Meets Target'] = kpi_df['Meets Target'].map({True: 'PASS', False: 'FAIL'})

# KPI Summary
sep = '-' * 52
print(f"{'KPI Summary':^52}")
print(sep)
print(f"  {'KPI':<28} {'Value':>6}  {'Target':>7}  Status")
print(sep)
for _, r in kpi_df.iterrows():
    print(f"  {r['KPI']:<28} {r['Value (%)']:>6}  {r['Target']:>7}  {r['Meets Target']}")
print(sep)
kpi_df[['KPI', 'Value (%)', 'Target', 'Meets Target']]

### Interpretation

**Validity Error Rate (36.4%)** is the biggest concern – over a third of records fail at least one validity check, which suggests no input validation exists at the source.

**Duplication Rate (9.1%)** is also serious. Even one duplicate in a transaction table means double-counted revenue.

**Completeness Rate (96%)** just misses the 98% target. The two missing critical fields (`customer_id` and `amount`) are the reason – both would break any revenue calculation.

**Format Consistency (81.8%)** is the least urgent but still a problem. Mixed date formats and inconsistent casing mean any downstream processing needs to handle multiple variants.

## Task 4 – Validation Rules

In [ ]:
rule_customer_id_required = is_missing(df['customer_id'])
rule_amount_non_negative  = (amount_num < 0) | amount_num.isna()
rule_date_valid           = date_parsed.isna()
rule_currency_iso4217     = invalid_currency
rule_txn_id_unique        = df.duplicated(subset=['transaction_id'], keep=False)

rules = {
    'customer_id_required':  rule_customer_id_required,
    'amount_non_negative':   rule_amount_non_negative,
    'date_valid':            rule_date_valid,
    'currency_iso4217':      rule_currency_iso4217,
    'transaction_id_unique': rule_txn_id_unique,
}

val_summary = pd.DataFrame({
    'rule_name':      list(rules.keys()),
    'affected_rows':  [int(m.sum()) for m in rules.values()],
    'violation_rate': [f'{m.mean()*100:.1f}%' for m in rules.values()],
}).sort_values('affected_rows', ascending=False).reset_index(drop=True)
val_summary

In [ ]:
# Records that fail at least one rule
any_violation = pd.Series(False, index=df.index)
for mask in rules.values():
    any_violation = any_violation | mask

flagged = df[any_violation].copy()
flagged['violation_flags'] = [
    ', '.join(name for name, mask in rules.items() if mask[i])
    for i in flagged.index
]
print(f'Flagged records: {len(flagged)} / {len(df)}')
flagged

## Task 5 – Audit Summary

In [ ]:
audit_summary = pd.DataFrame([
    ['Missing customer_id',                        1, 'High',   'Retrieve from source system; block from revenue reports until resolved'],
    ['Duplicate transaction record (T0006)',        2, 'High',   'De-duplicate on transaction_id keeping row with latest last_updated; investigate ETL for double-processing'],
    ['Negative amount (T0003: -35.00 USD)',         1, 'High',   'Classify as refund or data-entry error; create refund_type field or separate refunds table'],
    ['Impossible date (T0007: 2026-02-30)',         1, 'High',   'Correct via source system; add calendar-validity check at ingestion'],
    ['Missing amount (T0009)',                      1, 'High',   'Retrieve from payment processor logs; exclude from revenue aggregations until resolved'],
    ['Invalid currency code (T0005: EURO)',         1, 'Medium', 'Map EURO -> EUR via ISO 4217 lookup; enforce validation at ETL boundary'],
    ['Inconsistent date formats (T0002, T0003)',    2, 'Medium', 'Normalise all dates to ISO 8601 (YYYY-MM-DD) in ingestion layer'],
    ['Missing payment_method (T0010)',              1, 'Medium', 'Impute from customer transaction history or mark as unknown'],
    ['Extreme amount outlier (T0008: 1 000 000)',   1, 'Medium', 'Verify with payment processor; route to review queue before including in reports'],
    ['Case inconsistency – region/payment (T0002)', 1, 'Low',   'Normalise: region uppercase, payment_method lowercase in ETL transform'],
    ['Suspicious update lag > 60 days (T0006)',     2, 'Low',   'Investigate reason for late modification; document in correction log'],
], columns=['issue_type', 'affected_rows', 'severity', 'recommended_next_action'])
audit_summary

## Task 6 – Cleaning Steps for Next Week

Not implementing yet, but here's what needs to happen:

1. **De-duplicate** on `transaction_id` – sort by `last_updated` descending, then `drop_duplicates(keep='first')` to retain the most recent version of each transaction.

2. **Fix date formats** – parse everything with `pd.to_datetime(format='mixed')` and reformat to `YYYY-MM-DD`. Rows that still can't parse (like `2026-02-30`) should go to a quarantine table, not be silently dropped.

3. **Negative amounts** – don't just remove them. T0003's -35.00 is almost certainly a refund. The right fix is a `transaction_type` field or a separate refunds table before touching the values.

4. **Missing critical fields** – `customer_id` and `amount` need to come from the source system. If they can't be recovered, move those rows out of the main table and track them separately.

5. **Standardise categoricals** – currency to uppercase ISO 4217, region to uppercase ISO 3166, payment_method to lowercase. Best handled in the ETL before the data lands here, not patched after.

6. **Outlier review** – T0008's 1,000,000 EUR amount needs manual sign-off. Use IQR-based flagging and route to a review queue rather than auto-removing.

## Reflection

**1. Strongest KPI signal?**  
Validity Error Rate at 36.4%. More than a third of rows have a validity problem – that's not random noise, it means there are no input checks at the ingestion point.

**2. What to escalate first?**  
The T0006 duplicate. A duplicate transaction ID means double-counted revenue, and it likely points to an ETL bug that could be affecting more records outside this sample.

**3. What extra metadata would help?**  
A customer master table would let me check referential integrity – are all `customer_id` values actually valid? An ETL job log would also help trace where each record came from and when, which would make root-cause analysis much easier.

## Bonus – Reusable Audit Helper Function

In [ ]:
def run_validation_rules(df, rules):
    """
    Takes a dataframe and a dict of named boolean masks (True = violation)
    and returns a sorted summary of violations per rule.
    """
    records = []
    for name, mask in rules.items():
        records.append({
            'rule_name':       name,
            'affected_rows':   int(mask.sum()),
            'violation_rate':  f'{mask.mean() * 100:.1f}%',
            'example_indices': df.index[mask].tolist()[:3],
        })
    return (
        pd.DataFrame(records)
        .sort_values('affected_rows', ascending=False)
        .reset_index(drop=True)
    )


run_validation_rules(df, rules)

### Notes on the function

`df` can be the full dataframe or a filtered slice – passing `df[df['region'] == 'DE']` for example scopes the violation counts to just German transactions without changing the function at all.

`rules` is just a dict of name → boolean mask, so adding or removing a check is as simple as adding or removing a key. Keeping it separate from the function means the same function can be reused with different rule sets.

Neither parameter has a default because both are required to produce any output.

## Submission Checklist

- [x] Dataset described
- [x] Issues mapped to 5 dimensions (Completeness, Uniqueness, Validity, Consistency, Integrity)
- [x] 4 KPIs with targets and interpretation
- [x] 5 validation rules with violation counts
- [x] Audit summary with severity and next actions
- [x] 6 cleaning steps proposed (not implemented)
- [x] Bonus helper function with parameter notes